In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import colorir as cl
import numpy as np
from analyses import io

In [ ]:
colors = cl.StackPalette.load("safe")

In [ ]:
celldf = io.read_celldfs(
    "../runs/invading/", 
    levels=["replica", "cell_energy", "mul_energy"]
).with_columns(
    pl.col("cell_energy").str.strip_prefix("cell_energy-").cast(pl.UInt32),
    pl.col("mul_energy").str.strip_prefix("mul_energy-").cast(pl.UInt32),
    pl.col("replica").cast(pl.UInt32),
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5,
).with_columns(
    mul_gamma=20 - pl.col("mul_energy"),
    cell_gamma=20 - pl.col("cell_energy"),
)
celldf

In [ ]:
grouppers = ["mul_gamma", "cell_gamma"]
clusterdf = celldf.filter(pl.col("wtime") >= 4e6).group_by(grouppers + ["lineage"]).agg(
    cluster_x=pl.col("center_x").mean(),
    cluster_y=pl.col("center_y").mean(),
    cluster_displ=pl.col("displ").mean()
).sort(grouppers)
clusterdf

In [ ]:
pvdf = clusterdf.filter(
    pl.col("mul_gamma") < 12
).pivot(
    on="mul_gamma", 
    index=["cell_gamma", "lineage"], 
    values="cluster_displ"
).sort("cell_gamma", "lineage")
pvdf

In [ ]:
x = pvdf["cell_gamma"].unique()
y = pvdf.columns[2:]
dropped = pvdf.drop("cell_gamma", "lineage").to_numpy()
ma = dropped[::2].T
pa = dropped[1::2].T
fig = go.Figure(go.Heatmap(
    z=pa - ma,
)).update_layout(
    template="plotly_white",
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    xaxis_constrain="domain",
    xaxis_tickvals=np.arange(len(x)),
    xaxis_ticktext=x,
    yaxis_tickvals=np.arange(len(y)),
    yaxis_ticktext=y
)
fig

In [ ]:
filterdf = celldf.filter(
    pl.col("wtime") >= 4e6,
    mul_energy=12
)
fig = px.violin(
    filterdf.filter(lineage="mul"),
    x="cell_gamma",
    y="displ",
    color="lineage",
    color_discrete_sequence=colors
).update_traces(
    jitter=1,
    marker_line_width=1,
    marker_line_color="white",
    marker_opacity=0.5
).add_traces(
    px.scatter(
        filterdf.filter(lineage="mul").group_by(
            "cell_gamma", 
            "lineage",
            maintain_order=True
        ).mean().with_columns(
            x=pl.col("cell_gamma")# .cast(pl.Int32) + pl.when(pl.col("lineage") == "mul").then(-0.18).otherwise(0.18)
        ),
        x="x",
        y="displ",
        color="lineage",
        color_discrete_sequence=colors
    ).data
).update_layout(
    template="plotly_white",
    width=450,
    height=300,
    showlegend=False,
    xaxis_dtick=1,
    # yaxis_range=[0, max_chem],
    yaxis_title="distance to peak"
)
# io.save_plot(fig, "../plots/steady-mixed-pop")
fig

In [ ]:
diffdf = filterdf.group_by(
    "cell_gamma", 
    "lineage",
    maintain_order=True
).mean().sort("cell_gamma")
diff = diffdf.filter(lineage="uni")["displ"] - diffdf.filter(lineage="mul")["displ"]
px.line(
    y=diff,
    x=diffdf["cell_gamma"].unique()
).update_layout(
    template="plotly_white",
    width=400,
    height=300,
    xaxis_title="cell_gamma",
    yaxis_title="mean_p - mean_m",
    xaxis_dtick=1
)

In [ ]:
from statsmodels.stats.weightstats import ttest_ind
filterdf = celldf.filter(
    pl.col("wtime") >= 4e6,
    mul_energy=12,
    cell_gamma=7,
)
_, p, _ = ttest_ind(
    filterdf.filter(
        lineage="mul"
    )["displ"], 
    filterdf.filter(
        lineage="uni"
    )["displ"]
)
p